In [3]:
import pandas as pd
import numpy as np
import glob
import os

In [4]:
cleaned_files = glob.glob("../data/cleaned/*.csv")

print("Number of cleaned files:", len(cleaned_files))

Number of cleaned files: 8


In [6]:
for file in cleaned_files:

    df = pd.read_csv(file)

    print("\n" + "=" * 60)
    print(os.path.basename(file))
    print("=" * 60)

    print(df["Label"].value_counts())

    del df


Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX_cleaned.csv
Label
BENIGN      123295
PortScan     90819
Name: count, dtype: int64

Friday-WorkingHours-Afternoon_cleaned.csv
Label
DDoS      128016
BENIGN     95096
Name: count, dtype: int64

Friday-WorkingHours-Morning.pcap_ISCX_cleaned.csv
Label
BENIGN    182192
Bot         1953
Name: count, dtype: int64

Monday-WorkingHours.pcap_ISCX_cleaned.csv
Label
BENIGN    502983
Name: count, dtype: int64

Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX_cleaned.csv
Label
BENIGN          252936
Infiltration        36
Name: count, dtype: int64

Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX_cleaned.csv
Label
BENIGN                        162157
Web Attack � Brute Force        1470
Web Attack � XSS                 652
Web Attack � Sql Injection        21
Name: count, dtype: int64

Tuesday-WorkingHours.pcap_ISCX_cleaned.csv
Label
BENIGN         412692
FTP-Patator      5933
SSH-Patator      3219
Name: count, dtype: int64

Wednesday-workin

First we will be doing binary classification , for that we will use 0/1 
where 0 = normal traffic, 1 = attack

In [6]:
for file in cleaned_files:

    df = pd.read_csv(file)

    # Create binary target column
    df["Target"] = (df["Label"] != "BENIGN").astype(int)

    print("\n", os.path.basename(file))
    print(df["Target"].value_counts())

    del df


 Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX_cleaned.csv
Target
0    123295
1     90819
Name: count, dtype: int64

 Friday-WorkingHours-Afternoon_cleaned.csv
Target
1    128016
0     95096
Name: count, dtype: int64

 Friday-WorkingHours-Morning.pcap_ISCX_cleaned.csv
Target
0    182192
1      1953
Name: count, dtype: int64

 Monday-WorkingHours.pcap_ISCX_cleaned.csv
Target
0    502983
Name: count, dtype: int64

 Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX_cleaned.csv
Target
0    252936
1        36
Name: count, dtype: int64

 Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX_cleaned.csv
Target
0    162157
1      2143
Name: count, dtype: int64

 Tuesday-WorkingHours.pcap_ISCX_cleaned.csv
Target
0    412692
1      9152
Name: count, dtype: int64

 Wednesday-workingHours.pcap_ISCX_cleaned.csv
Target
0    417035
1    193759
Name: count, dtype: int64


In [7]:
all_data = []

for file in cleaned_files:

    df = pd.read_csv(file)

    # Create binary target
    df["Target"] = (df["Label"] != "BENIGN").astype(int)

    all_data.append(df)

    print(
        os.path.basename(file),
        "→",
        df.shape
    )

combined_df = pd.concat(all_data, ignore_index=True)

del all_data

print("\nCombined shape:", combined_df.shape)

Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX_cleaned.csv → (214114, 80)
Friday-WorkingHours-Afternoon_cleaned.csv → (223112, 80)
Friday-WorkingHours-Morning.pcap_ISCX_cleaned.csv → (184145, 80)
Monday-WorkingHours.pcap_ISCX_cleaned.csv → (502983, 80)
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX_cleaned.csv → (252972, 80)
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX_cleaned.csv → (164300, 80)
Tuesday-WorkingHours.pcap_ISCX_cleaned.csv → (421844, 80)
Wednesday-workingHours.pcap_ISCX_cleaned.csv → (610794, 80)

Combined shape: (2574264, 80)


Separate X and Y

In [8]:
X = combined_df.drop(columns=["Label", "Target"])
y = combined_df["Target"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2574264, 78)
y shape: (2574264,)


don't want to blindly give all 78 columns to the model. checking whether any columns still contain problematic values.

In [9]:
print("Missing values:", X.isnull().sum().sum())

print(
    "Infinite values:",
    np.isinf(X.select_dtypes(include="number")).sum().sum()
)

print("\nData types:")
print(X.dtypes.value_counts())

Missing values: 0
Infinite values: 0

Data types:
int64      54
float64    24
Name: count, dtype: int64


Before splitting the data, let's see how many normal vs attack records we have overall.

In [10]:
print("Target distribution:")
print(y.value_counts())

print("\nTarget percentage:")
print(y.value_counts(normalize=True) * 100)

Target distribution:
Target
0    2148386
1     425878
Name: count, dtype: int64

Target percentage:
Target
0    83.45632
1    16.54368
Name: proportion, dtype: float64


Model Training

In [11]:
from sklearn.model_selection import train_test_split

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (2059411, 78)
X_test : (514853, 78)
y_train: (2059411,)
y_test : (514853,)


In [13]:
print(X_train.describe().T[["min", "max"]].head(15))

                                     min           max
Destination Port                     0.0  6.553500e+04
Flow Duration                      -13.0  1.200000e+08
Total Fwd Packets                    1.0  2.197590e+05
Total Backward Packets               0.0  2.919220e+05
Total Length of Fwd Packets          0.0  1.290000e+07
Total Length of Bwd Packets          0.0  6.554530e+08
Fwd Packet Length Max                0.0  2.482000e+04
Fwd Packet Length Min                0.0  2.293000e+03
Fwd Packet Length Mean               0.0  5.939286e+03
Fwd Packet Length Std                0.0  7.125597e+03
Bwd Packet Length Max                0.0  1.737600e+04
Bwd Packet Length Min                0.0  2.896000e+03
Bwd Packet Length Mean               0.0  5.800500e+03
Bwd Packet Length Std                0.0  8.194660e+03
Flow Bytes/s                -261000000.0  2.071000e+09


In [14]:
negative_counts = (X_train < 0).sum()

print(negative_counts[negative_counts > 0])

Flow Duration                   86
Flow Bytes/s                    67
Flow Packets/s                  86
Flow IAT Mean                   86
Flow IAT Max                    86
Flow IAT Min                  2305
Fwd IAT Min                     16
Fwd Header Length               30
Bwd Header Length               19
Fwd Header Length.1             30
Init_Win_bytes_forward      761315
Init_Win_bytes_backward    1011855
min_seg_size_forward            30
dtype: int64


In [15]:
for col in negative_counts[negative_counts > 0].index:
    print("\n", col)
    print(X_train.loc[X_train[col] < 0, col].value_counts().head(10))


 Flow Duration
Flow Duration
-1     82
-2      1
-13     1
-12     1
-4      1
Name: count, dtype: int64

 Flow Bytes/s
Flow Bytes/s
-1.200000e+07    42
-6.000000e+06    11
-8.000000e+06    10
-4.615385e+05     1
-6.666667e+05     1
-2.610000e+08     1
-1.930000e+08     1
Name: count, dtype: int64

 Flow Packets/s
Flow Packets/s
-2.000000e+06    82
-1.000000e+06     1
-1.538462e+05     1
-1.666667e+05     1
-5.000000e+05     1
Name: count, dtype: int64

 Flow IAT Mean
Flow IAT Mean
-1.0     82
-2.0      1
-13.0     1
-12.0     1
-4.0      1
Name: count, dtype: int64

 Flow IAT Max
Flow IAT Max
-1     82
-2      1
-13     1
-12     1
-4      1
Name: count, dtype: int64

 Flow IAT Min
Flow IAT Min
-1     2194
-2       24
-12      24
-3       14
-4       13
-13      11
-11      10
-5        9
-10       3
-8        1
Name: count, dtype: int64

 Fwd IAT Min
Fwd IAT Min
-1     14
-12     1
-8      1
Name: count, dtype: int64

 Fwd Header Length
Fwd Header Length
-83885125      3
-167770490 

In [16]:
negative_summary = {}

for col in X_train.columns:

    count = (X_train[col] < 0).sum()

    if count > 0:
        negative_summary[col] = {
            "negative_count": count,
            "negative_percentage": (count / len(X_train)) * 100,
            "minimum": X_train[col].min()
        }

negative_summary_df = pd.DataFrame(negative_summary).T

negative_summary_df.sort_values(
    "negative_count",
    ascending=False
)

,negative_count,negative_percentage,minimum
Init_Win_bytes_backward,1011855.0,49.133223,-1.000000e+00
Init_Win_bytes_forward,761315.0,36.967609,-1.000000e+00
Flow IAT Min,2305.0,0.111925,-1.400000e+01
Flow Duration,86.0,0.004176,-1.300000e+01
Flow Packets/s,86.0,0.004176,-2.000000e+06
Flow IAT Mean,86.0,0.004176,-1.300000e+01
Flow IAT Max,86.0,0.004176,-1.300000e+01
Flow Bytes/s,67.0,0.003253,-2.610000e+08
Fwd Header Length,30.0,0.001457,-3.221223e+10
Fwd Header Length.1,30.0,0.001457,-3.221223e+10


For our first model, let's treat all negative values in these affected features as missing values (NaN), and then fill them using the median calculated only from the training data.

In [17]:
X_train = X_train.copy()
X_test = X_test.copy()

In [20]:
negative_columns = [
    col for col in X_train.columns
    if (X_train[col] < 0).any()
]

print("Columns with negative values:")
print(negative_columns)
print(len(negative_columns))

Columns with negative values:
['Flow Duration', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Min', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Header Length.1', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'min_seg_size_forward']
13


In [21]:
for col in negative_columns:
    X_train.loc[X_train[col] < 0, col] = np.nan
    X_test.loc[X_test[col] < 0, col] = np.nan

In [22]:
for col in negative_columns:
    median_value = X_train[col].median()
    
    X_train[col] = X_train[col].fillna(median_value)
    X_test[col] = X_test[col].fillna(median_value)

In [23]:
print("Negative values in X_train:",
      (X_train < 0).sum().sum())

print("Negative values in X_test:",
      (X_test < 0).sum().sum())

print("Missing values in X_train:",
      X_train.isnull().sum().sum())

print("Missing values in X_test:",
      X_test.isnull().sum().sum())

Negative values in X_train: 0
Negative values in X_test: 0
Missing values in X_train: 0
Missing values in X_test: 0


ML model: Random Forest.

In [24]:
from sklearn.ensemble import RandomForestClassifier

In [25]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

In [26]:
model.fit(X_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

let's save the trained model so if your notebook/kernel crashes we dont have to train and waste time

In [27]:
import joblib

joblib.dump(model, "../model_random_forest.pkl")

print("Model saved successfully!")

Model saved successfully!


In [28]:
y_pred = model.predict(X_test)

In [29]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy (%):", accuracy * 100)

Accuracy: 0.9985452158188842
Accuracy (%): 99.85452158188842


In [30]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred,
    target_names=["BENIGN", "ATTACK"]
))

              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00    429677
      ATTACK       1.00      0.99      1.00     85176

    accuracy                           1.00    514853
   macro avg       1.00      1.00      1.00    514853
weighted avg       1.00      1.00      1.00    514853



In [31]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[429369    308]
 [   441  84735]]


In [32]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

Accuracy : 0.9985452158188842
Precision: 0.9963783027409663
Recall   : 0.9948224852071006
F1 Score : 0.995599786157832
